In [ ]:
# 交叉熵
import numpy as np
def cross_entropy(logits, target):
    exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    softmax_probs = exps / np.sum(exps, axis=1, keepdims=True)
    n = logits.shape[0]
    correct_logprobs = -np.log(softmax_probs[range(n),target] + 1e-12)
    loss = np.sum(correct_logprobs) / n
    return loss

def kmeans(x, k, max_iters=100, tol=1e-4):
    n_samples, n_features = x.shape
    indices = np.random.choice(n_samples, k, replace=False)
    centroids = x[indices]

    for i in range(max_iters):
        distances = np.linalg.norm(x[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)
        new_centroids = np.array([x[labels==j].mean(axis=0) if len(x[labels==j]) > 0 else centroids[j] for j in range(k)])
        if np.linalg.norm(new_centroids - centroids) < tol:
            print(f"Converged at iteration {i}")
            break
        centroids = new_centroids
    return centroids, labels

def AUC(y_score, y_true):
    indices = sorted(range(len(y_score)), key=lambda i:y_score[i])
    sorted_labels = [y_true[i] for i in indices]
    pos_c = np.sum(y_true)
    neg_c = len(y_true) - pos_c
    rank_sum = 0
    for i in range(len(sorted_labels)):
        if sorted_labels[i] == 1:
            rank_sum += (i+1)
    auc = (rank_sum - pos_c * (pos_c + 1)/2)/(pos_c * neg_c)
    return auc

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
class MHA_cache(nn.Module):
    def __init__(self, d_models, n_heads=8, dropout=0.1, bias=True):
        super().__init__()
        assert d_models % n_heads == 0
        self.d_k = d_models // n_heads
        self.n_heads = n_heads

        self.W_q = nn.Linear(d_models, d_models, bias=bias)
        self.W_k = nn.Linear(d_models, d_models, bias=bias)
        self.W_v = nn.Linear(d_models, d_models, bias=bias)
        self.W_o = nn.Linear(d_models, d_models, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False, mask=None):
        B,L,D = x.shape
        Q = self.W_q(x).view(B,L,self.n_heads, self.d_k).transpose(1,2)
        K = self.W_k(x).view(B,L,self.n_heads, self.d_k).transpose(1,2)
        V = self.W_v(x).view(B,L,self.n_heads, self.d_k).transpose(1,2)

        if past_kv is not None:
            past_k, past_v = past_kv
            K = torch.cat([past_k, K], dim=2)
            V = torch.cat([past_v, V], dim=2)
        present_kv = (K,V) if use_cache else None

        score = torch.matmul(Q, K.transpose(-2,-1))*torch.rsqrt(self.d_k)
        if mask is not None:
            score = score.masked_fill(mask==0, float('-inf'))
        attn = F.softmax(score, dim=-1)
        attn = self.dropout(attn)
        output = torch.matmul(attn, V).transpose(1,2).contiguous().view(B,L,D)
        output = self.W_o(output)
        return output, present_kv
        